# Batch deposit driver

The pipeline lives in `levy_curate/`; this notebook only drives it and hosts QA.

1. **Scaffold** -- build the complete metadata table from files + a partial CSV
2. **Validate** -- catch every problem findable without the network
3. **Plan** -- dry run: what would be created
4. **Deposit** -- create datasets and upload, recording a resumable manifest
5. **QA** -- harvest file DOIs, re-detect types, then publish in the UI

Everything batch-specific is in the config YAML. Edit that, not this notebook.

In [ ]:
from levy_curate import BatchConfig
from levy_curate import scaffold, inventory, deposit, harvest
from levy_curate.manifest import Manifest

cfg = BatchConfig.from_yaml("configs/geotiffs.yaml")
print(cfg.summary())

## 1. Scaffold

Reads every file in `files_dir` and merges the partial CSV (filename + raw OCHRE
link). Enumeration starts from disk, so a file missing from the CSV still gets a
row -- flagged rather than silently dropped.

Repeat a filename across rows in the partial CSV to attach several links to one
file; they collapse into the `OCHRE link(s):` format.

In [ ]:
result = scaffold.build(cfg)
print(result.report())

In [ ]:
# Files still needing an OCHRE link. Add them to the partial CSV and re-run
# the cell above until this is empty.
result.without_links

## 2. Validate

Runs before any network call. Checks required columns, empty values, duplicate
filenames, one dataset_description per dataset, description length and embedded
newlines (the Bean Validation failure in errors.md), and reconciles every row
against the files on disk.

In [ ]:
df = inventory.load(cfg)
report = inventory.validate(df, cfg)
print(report.report())

In [ ]:
# Rows whose file is missing from disk
report.missing_on_disk

In [ ]:
# Files on disk that no row mentions -- these would NOT be deposited
report.missing_in_table

## 3. Plan

Dry run. Creates nothing; prints the datasets, their file counts, and the target
installation. Check the target line before proceeding.

In [ ]:
print(deposit.plan(df, cfg))

## 4. Deposit

Writes to the manifest as it goes. If a dataset fails, fix the cause and re-run
this cell -- datasets already uploaded are skipped, not duplicated.

Set `DATAVERSE_TARGET=harvard` in `.env` only after a clean demo run.

In [ ]:
assert report.ok, "inventory is not valid -- fix the errors above first"

man = deposit.deposit(df, cfg)
print()
print(man.report())

In [ ]:
# {dataset_title: pid} -- the successor to upload_pids.json
man.pids()

## 5. QA

Review the drafts in the Dataverse UI, then harvest file DOIs and fix any
mis-detected file types. Publish from the UI once satisfied.

In [ ]:
dois = harvest.harvest_batch(cfg)
dois.head()

In [ ]:
# Dry run first: shows current content types without changing anything.
for title, pid in man.pids().items():
    print(title)
    display(harvest.redetect(pid, cfg.credentials, dry_run=True).head())

In [ ]:
# Apply re-detection (set dry_run=False deliberately)
# for title, pid in man.pids().items():
#     harvest.redetect(pid, cfg.credentials, dry_run=False)